# 4.1 — Core Pipeline Demo (with ByteTrack)

Demonstrate the **real-time PPE compliance pipeline** with ByteTrack:

1. **YOLOv8 detection** — detect person, helmet, no-helmet, no-vest, vest
2. **ByteTrack** — persistent person IDs across frames, re-identification
3. **PPE–worker association** — IoU-based matching of PPE items to nearest person
4. **Compliance classification** — Safe / Violation per worker
5. **Zone-based people counting** — count workers in configurable zones
6. **Visualisation** — annotated output with track IDs, bounding boxes, status, HUD

## Setup

In [ ]:
# @title Install dependencies
!pip install -q ultralytics roboflow loguru typer python-dotenv pyyaml matplotlib opencv-python-headless albumentations

In [ ]:
# @title Mount Drive or clone repo
import os
from pathlib import Path
import sys

!git clone https://github.com/Hndra04/AlGear
PROJECT_DIR = Path("/content/AlGear")

os.chdir(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project root: {PROJECT_DIR}")

In [ ]:
# @title Set Roboflow API key
import os
os.environ["ROBOFLOW_API_KEY"] = ""

from algear.config import ROBOFLOW_DIR, ROBOFLOW_API_KEY
print(f"API key set: {bool(ROBOFLOW_API_KEY)}")
print(f"Dataset at: {ROBOFLOW_DIR}")

In [ ]:
# @title Download dataset (if not already present)
if not ROBOFLOW_DIR.exists():
    from algear.dataset import download_roboflow
    download_roboflow(output_dir=ROBOFLOW_DIR)
else:
    print(f"Dataset already exists at {ROBOFLOW_DIR}")

In [ ]:
# @title Check GPU availability
import torch

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Initialise Pipeline

In [ ]:
# @title Load pipeline with ByteTrack enabled
from algear.core import CompliancePipeline, PipelineConfig

MODEL_PATH = PROJECT_DIR / "models" / "resplit-oversample-conservative" / "weights" / "best.pt"
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

config = PipelineConfig(
    conf=0.25,
    iou=0.45,
    imgsz=640,
    device=device,
    use_tracker=True,  # ByteTrack enabled
    tracker_cfg="bytetrack.yaml",
    zone_configs=[
        {"name": "left",  "polygon": [[0.0, 0.0], [0.5, 0.0], [0.5, 1.0], [0.0, 1.0]]},
        {"name": "right", "polygon": [[0.5, 0.0], [1.0, 0.0], [1.0, 1.0], [0.5, 1.0]]},
    ],
)

pipeline = CompliancePipeline(model_path=MODEL_PATH, config=config)
print(f"Pipeline ready — model loaded, ByteTrack ON, 2 zones defined")

## 2. Process Single Image

In [ ]:
# @title Pick a test image
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESPLIT_DIR = PROJECT_DIR / "data" / "processed" / "construction-site-safety-resplit"
TEST_IMAGE_DIR = RESPLIT_DIR / "test" / "images"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_images = sorted(f for f in TEST_IMAGE_DIR.iterdir() if f.suffix.lower() in IMAGE_EXTS)
print(f"Available test images: {len(test_images)}")

# Pick first image
img_path = test_images[0]
frame = cv2.imread(str(img_path))
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(frame_rgb)
plt.title(f"Input: {img_path.name} ({frame.shape[1]}x{frame.shape[0]})")
plt.axis("off")
plt.show()

In [ ]:
# @title Run pipeline on single image
result = pipeline.process_frame(frame, frame_idx=0)

print(f"\n=== Pipeline Results ===")
print(f"Image: {img_path.name} ({result.img_w}x{result.img_h})")
print(f"Inference time: {result.inference_ms:.1f} ms")
print(f"Detections: {len(result.detections)}")
print(f"Persons: {result.person_count}")
print(f"Compliant: {result.compliant_count}")
print(f"Violations: {result.violation_count}")
print(f"Compliance rate: {result.compliance_rate:.1%}")
print(f"Zone counts: {result.zone_counts}")

print(f"\nWorkers detail:")
for i, w in enumerate(result.workers):
    status = "SAFE" if w.is_compliant else "VIOLATION"
    tid = result.track_ids[i] if i < len(result.track_ids) else "-"
    ppe = []
    if w.has_helmet: ppe.append("helmet")
    if w.has_no_helmet: ppe.append("no-helmet")
    if w.has_vest: ppe.append("vest")
    if w.has_no_vest: ppe.append("no-vest")
    print(f"  Worker {i} (ID:{tid}): {status} — {', '.join(ppe)}")

In [ ]:
# @title Visualise annotated frame
annotated = pipeline.render(frame.copy(), result)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.title(f"PPE Compliance — {result.person_count} persons, {result.violation_count} violations")
plt.axis("off")
plt.show()

## 3. Batch Process Test Set

In [ ]:
# @title Process all test images
import time

batch_results = []
t_start = time.perf_counter()

for i, img_path in enumerate(test_images):
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    fr = pipeline.process_frame(frame, frame_idx=i)
    batch_results.append({"path": img_path.name, "result": fr})

    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(test_images)} images...")

elapsed = time.perf_counter() - t_start
avg_fps = len(batch_results) / elapsed if elapsed > 0 else 0

print(f"\nBatch complete: {len(batch_results)} images in {elapsed:.1f}s ({avg_fps:.1f} FPS)")

In [ ]:
# @title Batch results summary
total_persons = sum(r["result"].person_count for r in batch_results)
total_compliant = sum(r["result"].compliant_count for r in batch_results)
total_violations = sum(r["result"].violation_count for r in batch_results)
avg_inf = np.mean([r["result"].inference_ms for r in batch_results])

print(f"\n{'='*50}")
print(f"  BATCH PIPELINE RESULTS")
print(f"{'='*50}")
print(f"  Images processed:    {len(batch_results)}")
print(f"  Total persons:       {total_persons}")
print(f"  Total compliant:     {total_compliant}")
print(f"  Total violations:    {total_violations}")
print(f"  Compliance rate:     {total_compliant/total_persons:.1%}" if total_persons > 0 else "N/A")
print(f"  Avg inference:       {avg_inf:.1f} ms")
print(f"  Avg FPS:             {1000/avg_inf:.1f}")
print(f"{'='*50}")

In [ ]:
# @title Zone counting summary
from collections import defaultdict

zone_totals = defaultdict(int)
for r in batch_results:
    for zname, cnt in r["result"].zone_counts.items():
        zone_totals[zname] += cnt

print("\nZone counting across all test images:")
for zname, total in sorted(zone_totals.items()):
    print(f"  {zname}: {total} person-detections")

## 4. Sample Gallery

In [ ]:
# @title Show 6 annotated samples
sample_indices = np.linspace(0, len(test_images) - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, idx in zip(axes.flat, sample_indices):
    img_path = test_images[idx]
    frame = cv2.imread(str(img_path))
    fr = pipeline.process_frame(frame, frame_idx=idx)
    annotated = pipeline.render(frame.copy(), fr)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(f"{fr.person_count} persons, {fr.violation_count} violations")
    ax.axis("off")

plt.suptitle("Pipeline Demo — Sample Results", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Process Video (Optional)

Upload a video file or use a test video to see the pipeline in action.

In [ ]:
# @title Upload a video file (optional)
# Uncomment to upload and process a video:
# from google.colab import files
# uploaded = files.upload()  # upload your .mp4 file
# video_path = list(uploaded.keys())[0]
#
# results = pipeline.process_video(
#     source=video_path,
#     output_path=str(PROJECT_DIR / "output_annotated.mp4"),
#     show=False,
# )
#
# # Download the annotated video
# files.download(str(PROJECT_DIR / "output_annotated.mp4"))